# Deep Research Agent: Evidence-Graph Architecture

Author: Matt Achachlouei

This presentation walks through the design, implementation, and evaluation of a deep research system that treats research as **graph construction** rather than pipeline synthesis.

---

## 1. Problem Framing and Thesis

### The Problem
Retrieval assumes you know what you are looking for. Deep research assumes you do not. The shape of a research question changes as evidence accumulates. 

Most agents treat research as a **Pipeline**: `Query -> Search -> Summarize -> Synthesize`.

### The Fatal Failure Modes of Pipelines
1. **Premature Synthesis (FM-01):** The system stops searching when it has "enough text to write," not when it knows the literature is settled. It synthesizes toward the most available consensus, smoothing over critical minority views.
2. **Ungrounded Claims (FM-02):** Citations are retrofitted to an LLM's generated prose, leading to plausible but unsupported assertions.
3. **Opaque Confidence (FM-03):** The system relies on subjective "vibes-based" confidence from the LLM, treating a highly-contested theory and a universal consensus with equal rhetorical certainty.

### The Core Idea: Research as Graph Construction
Instead of synthesizing prose, this system builds an **Evidence Graph**.
- **Nodes** are atomic, source-tagged factual claims.
- **Edges** are typed logical relationships (`Supports`, `Contradicts`, `Qualifies`).
- **Contradictions** are not noise to be averaged out; they are the primary signal that drives targeted re-search.
Synthesis only occurs when the graph's topology reaches empirical stability.

## 2. Full System Design (Production Architecture)

### Agent Boundaries & Orchestration (LangGraph)
1. **Query Architect:** Decomposes vague user queries into structured research briefs and allocates search budgets based on difficulty.
2. **Search Agent:** Executes parallel searches across source hierarchies (arXiv, Web) and handles rate limits/fallbacks.
3. **Claim Extractor:** Decomposes raw documents into atomic, falsifiable claims with preserved provenance.
4. **Graph Manager:** Maintains the networkx state, runs pairwise edge classification, and computes structural confidence.
5. **Contradiction Hunter:** Identifies unresolved `Contradicts` edges and spawns targeted search queries to resolve them.
6. **Skeptic:** Red-teams the graph prior to stabilization, actively searching for disconfirming evidence against high-confidence claims.
7. **Report Generator:** Synthesizes the stable graph into a final structured markdown report.

### Memory & State Handling
- **Compression by Design:** We never pass 90,000 tokens of raw documents into a synthesis prompt. Documents are compressed into atomic claims (~7:1 ratio). The LLM synthesis step only reads the structured graph.
- **Strict Provenance:** Raw documents are discarded from memory after extraction, but the provenance chain (URL, date, credibility weight) lives permanently inside the Claim objects.

### Tooling & Failure Handling
- **No Silent Failures:** If `arxiv_search` fails, it falls back to `web_search`. If all fail, the sub-question is explicitly flagged as "low-evidence/search-failed" in the final report metadata rather than silently hallucinating.

### Quality & Reliability Mechanisms
- **Adversarial Gates:** The **Skeptic** prevents "source monoculture." If all evidence points one way, the Skeptic is forced to run a disconfirmation query to ensure the consensus isn't an artifact of incomplete retrieval.

## 3. Why This Architecture? (Tradeoffs & Rationale)

Every design choice exists to defeat one of the core failure modes.

| Component | Why it exists | Tradeoff / Cost | How it can fail |
| :--- | :--- | :--- | :--- |
| **Atomic Claims vs Document Summaries** | Whole documents hide contradictions. Atomization makes disagreements structurally findable. | Higher LLM cost to extract and compare pairwise ($O(N^2)$). | The LLM might fragment a single nuanced claim into two, creating false contradictions. |
| **Contradiction Hunter** | Prevents "averaging out" fault lines. Turns dead-ends into precision search queries. | Latency. Resolving a debate requires multiple sequential loop iterations. | If the search index lacks the resolving paper, the contradiction remains irreducible (though surfacing it is still valuable). |
| **Graph Stability vs Fixed Iterations** | A fixed limit of 5 searches over-explores sparse topics and under-explores rich ones. Stability ensures completeness. | Unpredictable compute boundaries (capped by a hard maximum guardrail). | Graph can stabilize on a "comfortable but wrong" answer if search is systematically blind to a source type. |
| **Structural vs LLM Confidence** | LLMs are inherently sycophantic and overconfident. We calculate confidence via network topology (Credibility-weighted Support / Total Edges). | Ignores the "nuance" of the text in favor of raw edge counts. | Assumes our heuristic source weights (Peer-Reviewed=1.0, Web=0.4) are globally accurate. |

## 4. MVP Scope vs. Design Deferrals

**What was implemented (The ~2 day cut line):**
- End-to-end LangGraph orchestration.
- The core Evidence Graph data structure and structural confidence scoring.
- The Claim Extractor and Graph Manager (edge detection).
- The **Contradiction Hunter** (the system's "wow" factor).
- The Stability stopping condition.
- Real external tools (`Tavily`, `ArXiv`).

**What was intentionally deferred (and why):**
- *Semantic Edge Pre-filtering:* Currently doing naive $O(N^2)$ edge comparisons. In production, we'd use vector clustering to only compare claims in the same neighborhood. Excluded to focus on the logic, not the infra.
- *The Full Skeptic Debate Sub-graph:* The MVP Skeptic runs basic bias detection and single-shot disconfirmation, but doesn't spawn nested debate graphs. Kept simple to hit the 2-day timeline.
- *Fine-tuned Extractor:* Claim extraction is currently prompted. It needs a fine-tuned SLM for reliability.

## 5. Demo: Resolving the CoT Debate
Let's see the system tackle a query with known contradictory literature: 
*"Is chain-of-thought prompting an effective reasoning strategy for LLMs, or does it primarily improve output formatting?"*

In [ ]:
import sys
import os
from IPython.display import Markdown, display

sys.path.append(os.getcwd())

from src.demo.run_demo import run_demo
from src.graph.evidence_graph import EvidenceGraph

print("Starting Evidence Graph Research Session...\n")

# Run the demo (uses cached fixtures for reliability and quota management during presentation)
result = run_demo()

with open(result['report_path'], 'r') as f:
    report_content = f.read()

display(Markdown("### Final Research Report Snippet"))
report_lines = report_content.split('\n')
display(Markdown('\n'.join(report_lines[:25]) + "\n\n... [snip]"))

## 6. Evidence It Works (Graph Visualization)
The output strictly separates claims from evidence. Below is the underlying topological data structure that generated the report.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import json

def plot_evidence_graph_from_file(graph_path):
    with open(graph_path, 'r') as f:
        data = json.load(f)
    
    G = nx.node_link_graph(data)
    plt.figure(figsize=(10, 6))
    pos = nx.spring_layout(G, k=0.5, seed=42)
    
    edge_colors = []
    for u, v, d in G.edges(data=True):
        rel = d.get('relationship', '')
        if rel == 'supports': edge_colors.append('#2ecc71')
        elif rel == 'contradicts': edge_colors.append('#e74c3c')
        else: edge_colors.append('#95a5a6')
        
    nx.draw(G, pos, with_labels=False, node_color='#3498db', 
            node_size=800, edge_color=edge_colors, width=2, alpha=0.8)
    
    labels = {node: d.get('text', node)[:25] + '...' for node, d in G.nodes(data=True)}
    nx.draw_networkx_labels(G, pos, labels, font_size=8)
    
    plt.title("Evidence Graph Topology\n(Green: Supports | Red: Contradicts)")
    plt.show()

plot_evidence_graph_from_file(result['graph_path'])

## 7. Comparison / Ablation Study

We compared this system against a standard "Retrieve-and-Synthesize" pipeline on the Chain-of-Thought query.

| Metric | Baseline (Naive Pipeline) | Evidence Graph (Our System) |
| :--- | :--- | :--- |
| **Contradictions Surfaced** | 0 (Smoothed into "mixed results") | 3 explicitly flagged |
| **Resolution Behaviors** | N/A (Single shot) | 2 resolved via targeted search, 1 irreducible |
| **Citation Grounding** | Appended to paragraphs | Structurally linked to atomic claims |
| **Stopping Mechanism** | Arbitrary token limit | Graph Stability Threshold |

## 8. Failure Cases and Weaknesses

Where does the system break?

1. **Claim Fragmentation:** The LLM sometimes splits a nuanced assertion into two seemingly contradictory claims. If Claim A is *"CoT works well on 100B models"* and the extractor creates *"CoT works well"* and *"Requires 100B models"*, edge detection struggles.
2. **Edge Ambiguity:** The line between `Qualifies` and `Contradicts` is subjective. A paper showing CoT fails on math tasks *qualifies* the original CoT paper, but the LLM often flags it as a *contradiction*.
3. **Search Completeness Horizon:** If a contradiction resolving paper is paywalled or poorly SEO-optimized, the Contradiction Hunter exhausts its budget and marks it `irreducible`. The system only knows what it can ingest.
4. **Cost & Latency:** $O(N^2)$ edge checks per iteration is computationally aggressive for production without semantic pruning.

## 9. Self-Evaluation Methodology

How do we know a generated report is actually good? We evaluate on proxy metrics since ground truth doesn't exist.

1. **Citation Accuracy (Automated):** Ratio of claims in the report strictly matching their attached Source Nodes in the graph. (Target: 100%, achievable due to structural constraints).
2. **Contradiction Detection Rate (Synthetic):** Feed the system a curated corpus containing 5 known contradictory paper pairs. Does it surface all 5? (Currently ~80% due to extraction drops).
3. **Confidence Calibration:** High-confidence claims should empirically map to widely replicated studies. We test this by running the system on historically settled debates.
4. **The "Plausible vs. Correct" Filter:** A plausible system writes smooth prose. A correct system writes jagged prose explicitly stating *"Source A claims X, but Source B failed to replicate."* Our success metric explicitly rewards surfacing unresolved tension.

## 10. Research Context & Literature Positioning in Support of Design Decisions

1. Paper 1 — Search-R1 (arXiv:2503.09516). Jin et al 2025: Search-R1: Training LLMs to Reason and Leverage Search Engines with Reinforcement Learning. March 2025. URL: https://arxiv.org/abs/2503.09516
    - it is the architectural template for our Contradiction Hunter (§6.5). 
    - Search-R1 establishes the ReAct-style loop of interleaved reasoning and targeted retrieval that our FM-01 and QR-03 requirements depend on. 
    - Its outcome-only reward is also the gap our Evidence Graph is designed to fill...
        - the graph supplies the dense step-level supervision signal Search-R1 lacks (see the argument we make in the Appendix)

2. Paper 2 — SCoRe (arXiv:2409.12917). Kumar et al 2024: Training Language Models to Self-Correct via Reinforcement Learning. September 2024 (Google DeepMind). URL: https://arxiv.org/abs/2409.12917
    - it is the empirical justification for our Skeptic (§6.6) being a structured architectural component rather than a prompt-based critic. 
    - SCoRe's central finding — that SFT-based self-correction collapses to no-ops and requires multi-turn RL to work — is precisely why QR-02 is satisfied by a dedicated Skeptic with its own disconfirmation search rather than by asking the Report Generator to "be critical." 
    - It also supports our Appendix claim that architectural self-correction now is exactly what SCoRe proves we should eventually train.

3. Paper 3 — Test-Time Compute Scaling (arXiv:2408.03314). Snell et al 2024. Scaling LLM Test-Time Compute Optimally can be More Effective than Scaling Model Parameters. August 2024 (Google DeepMind). URL: https://arxiv.org/abs/2408.03314
    - This provides the empirical foundation for two specific design decisions: 
        - the Query Architect's expected_difficulty field driving adaptive search budget (§6.1) and 
        - graph stability as a content-driven stopping condition satisfying QR-03 (§6.4, §13). 
    - Snell et al. prove that matching compute to problem difficulty lets a smaller model outperform one 14× larger...
        - the same principle our system applies at the search-iteration level. 

4. Paper 4 — DRAGged into Conflicts (arXiv:2506.08500). Cattan et al 2025. DRAGged into Conflicts: Detecting and Addressing Conflicting Sources in Search-Augmented LLMs. June 2025 (Google Research). URL: https://arxiv.org/abs/2506.08500
    - This directly validates our design document's core epistemic commitment: 
        - QR-01 and the entire Contradiction Hunter mechanism. 
    - DRAGged's taxonomy of conflict types (factual contradictions, freshness/outdated, controversial opinions, misinformation) maps one-to-one onto our typed edges (supports/contradicts/qualifies/extends). 
    - The paper's empirical finding that LLMs resolve conflicts poorly by default but improve when prompted to reason about conflict type is the strongest published evidence for why our Evidence Graph promotes contradictions to first-class structural elements rather than letting synthesis smooth them over.

5. Paper 5 — GraphRAG (arXiv:2404.16130). Edge et al 2024. From Local to Global: A Graph RAG Approach to Query-Focused Summarization. April 2024 (Microsoft Research). URL: https://arxiv.org/abs/2404.16130
    - It is the strong precedent for our core architectural reframing in §4: 
        - research as graph construction rather than linear pipeline. 
    - GraphRAG empirically demonstrates that structured graphs over source material outperform flat retrieval for comprehensive multi-source synthesis. 
        - this is the exact problem class our FM-01 and FM-02 are designed around. 
    - Where it actually differs from our design:
        - GraphRAG builds an entity-relationship graph (entities as nodes, relationships as edges) for query-focused summarization of a fixed corpus. 
        - Our Evidence Graph is a claim graph (atomic claims as nodes, epistemic relationships as edges) built dynamically during research. 
        - These are structurally similar but epistemically different:
            - GraphRAG's edges represent factual relationships ("person X works at company Y")
            - Our edges represent epistemic relationships ("claim A contradicts claim B")
        - "isn't this just GraphRAG?" 
            - GraphRAG builds an entity graph over a known corpus; 
            - the Evidence Graph builds a claim graph over sources retrieved during research, with typed epistemic edges that drive exploration via contradiction. 
            - GraphRAG is closed-world and retrieval-focused; Ours is open-world and exploration-focused.


## 11. Decision Log / Process

**What changed my mind during the build:**
- *Initial Idea:* Multi-agent debate (Agent A writes, Agent B critiques). 
- *Why I rejected it:* I tested it and hit sycophantic convergence immediately. Agents argued from priors, not facts. I realized the conflict had to be rooted in the *external sources*, not the agent personas.
- *Pivot:* Shifted to the Evidence Graph. Made the conflict structural. 
- *Tooling:* Swapped from raw `networkx` dictionaries to Pydantic/Dataclass schemas midway because the edge logic was getting too brittle to debug.

## 12. What's Next? (Scaling to Production)

If I had another week, here is the roadmap:

1. **Fine-Tuned Claim Extractor (Training):** Stop using prompts for extraction. Train an 8B model specifically to decompose text into atomic assertions. This fixes the system's biggest fragility.
2. **Process Reward Models (PRMs):** The Evidence Graph transitions provide perfect step-level signals. We can train a PRM to reward the `Contradiction Hunter` for queries that successfully resolve edges, replacing our heuristic logic with RL.
3. **Temporal Confidence Weighting:** In LLM research, a 2022 paper and a 2024 paper shouldn't carry the same weight if they contradict. Confidence scores must decay temporally.
4. **Vector Pre-Filtering (Infra):** Implement HNSW or Faiss to only run edge-detection LLM calls on claims within semantic proximity, dropping complexity from $O(N^2)$ to $O(N \log N)$.